# 03-2. Embedding과 Vector Retrieval

- 예상 시간: 110분
- 선수 실습: 03-1_document_chunking.ipynb
- 실습 난이도: 중급
- 핵심 기술: Embedding, Vector Store, 유사도 점수 해석, Top-k 검색, Metadata Filter
- 최종 산출물: 유사도 기반 검색 파이프라인과 검색 결과 CSV
- 버전: 학생용 완성본 (모든 코드가 완성되어 있습니다. 직접 실행해 결과를 확인하세요)

- 적용 데이터: `data/samples/sample_report5.docx`


## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Embedding이 텍스트를 벡터 공간에 표현하는 방식을 설명할 수 있다.
2. 코사인 유사도 점수의 의미와 한계를 설명하고 검색 순위에 활용할 수 있다.
3. Vector Store의 Top-k 검색과 Metadata Filter를 사용해 검색 파이프라인을 구성할 수 있다.
4. 검색어의 표현 방식(짧음/구체적, 동의어, 동일 키워드의 다른 의미)이 검색 결과에
   미치는 영향을 관찰할 수 있다.


## 2. 문제 상황

Notebook 03-1에서 문서를 Chunk로 나눴다면, 이제 질문이 들어왔을 때 그 많은 Chunk 중
어떤 것을 검색 결과로 돌려줄지 정해야 한다. `search_keyword()`처럼 정확히 같은
단어가 있는지만 확인하는 방식은 "재택근무"로 검색했을 때 "리모트워크"라는 단어를
쓴 문서를 찾지 못한다. Embedding은 텍스트를 벡터로 바꿔, 정확히 같은 단어가 아니어도
의미가 비슷하면 가까운 벡터로 표현되도록 한다. 이 Notebook은 Embedding과 코사인
유사도를 이용한 검색 흐름과 결과 해석 방법을 확인한다.


## 3. 핵심 개념

### 3.1 정의

Embedding은 텍스트를 고정된 차원의 실수 벡터로 변환하는 것이다. 이렇게 만들어진
벡터 공간에서는 의미가 비슷한 텍스트일수록 벡터 사이의 거리가 가깝다.

### 3.2 개념이 필요한 이유

키워드 검색은 문서에 질문과 똑같은 단어가 있어야만 그 문서를 찾을 수 있다.
그러나 사용자는 같은 의미를 다른 단어로 표현하는 경우가 많다("재택근무" vs
"리모트워크"). Embedding 기반 검색은 단어가 달라도 의미가 비슷하면 찾아낼 수 있어,
검색어 표현에 덜 민감한 검색이 가능해진다.

### 3.3 주요 구성요소

| 구성요소 | 의미 |
|---|---|
| Vector Store | 문서 Embedding과 metadata를 저장하고 유사도 검색을 수행하는 구성요소 |
| 문서/질문 Embedding | 문서와 질문을 같은 벡터 공간으로 변환한 결과 |
| 코사인 유사도 | 질문과 문서 벡터가 얼마나 비슷한지를 비교하는 순위 점수 |
| 순위(rank) | 코사인 유사도가 높은 순서대로 매긴 순번 |
| Top-k | 순위가 높은 상위 k개 결과만 선택하는 것 |
| Metadata Filter | `document_id`, `section` 등 조건으로 검색 후보군을 미리 좁히는 것 |

### 3.4 동작 과정

```text
샘플 문서 → Document 변환 → Vector Store에 저장
                                  ↓
질문 → similarity_search_with_relevance_scores(query, k, filter)
                                  ↓
      Vector Store가 Embedding·유사도·정렬·Top-k 처리
                                  ↓
                    검색 결과 DataFrame
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `get_chroma_store()` | 공통 경로에서 디스크 기반 Chroma Collection 생성·재사용 |
| `add_documents_if_empty()` | 기존 Collection이 비었을 때만 문서와 metadata 저장 |
| `similarity_search_with_relevance_scores(...)` | 질문 Embedding·유사도·정렬·Top-k 검색 |
| `make_metadata_filter()` | Vector Store에 전달할 Metadata Filter |
| `search()` | 검색 조건을 Vector Store에 전달하고 결과 형식을 변환 |
| `build_result_dataframe()` | 검색 결과 DataFrame 출력 |

### 3.6 유사 개념과의 차이

**반드시 교정해야 할 오해**

```text
유사도 점수 0.85
≠ 정답일 확률 85%
```

코사인 유사도는 질문 벡터와 문서 벡터가 얼마나 비슷한지를 비교하는 점수일 뿐,
그 문서가 질문에 대한 올바른 답을 담고 있다는 보장이 아니다. 유사도 점수는 순위를
매기는 데 쓰는 상대적 지표이지, 정답일 확률 같은 절대적 지표가 아니다.

**키워드 검색 vs Embedding 검색**

| 구분 | 키워드 검색 | Embedding 검색 |
|---|---|---|
| 판단 기준 | 문자열이 정확히 일치하는가 | 벡터 방향이 얼마나 비슷한가 |
| 동의어 인식 | 불가능 | 가능(모델 성능에 따라 다름) |
| 계산 비용 | 낮음 | Embedding 계산 비용 발생 |
| 결과 해석 | "포함되어 있다/없다"로 명확함 | 점수는 상대적 순위 신호일 뿐 |

### 3.7 사용 시점과 적용 조건

문서량이 많고 사용자가 다양한 표현으로 질문할 가능성이 높다면 Embedding 검색이
유리하다. 반대로 정확한 코드, ID, 고유명사처럼 정확히 일치해야 의미가 있는 검색은
키워드 검색이 더 적합할 수 있다.

### 3.8 한계와 주의사항

- 코사인 유사도 값 자체는 모델과 문서 집합에 따라 상대적이며, 절대적인 기준값으로
  삼을 수 없다.
- Metadata Filter를 너무 좁게 걸면 실제 정답이 담긴 문서가 후보군에서 아예
  제외될 수 있다.
- Top-k가 너무 작으면 관련 문서를 놓치고, 너무 크면 무관한 문서까지 함께
  전달되어 비용이 늘어난다.

### 3.9 자주 발생하는 오해

"유사도 점수가 높으면 그 문서가 정답"이라는 오해가 가장 흔하다. 실제로는 유사도
점수가 높아도 질문과 무관한 이유로 벡터가 가까워졌을 수 있고, 반대로 진짜 정답을
담은 문서의 점수가 더 낮게 나올 수도 있다. 유사도 점수는 후보를 좁히는 신호일
뿐이며, 실제로 정답인지는 문서 본문을 확인해야 한다.

### 3.10 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store가 Embedding 저장, 유사도 계산, 정렬, Top-k를 처리하므로 애플리케이션에서
  같은 기능을 다시 구현할 필요가 없다.
- 유사도 점수는 검색 순위를 위한 상대적 지표이며, 정답일 확률이 아니다.
- Metadata Filter는 유사도 검색 전에 후보군 자체를 좁히는 절차다.
- 검색 결과의 실제 정확성은 유사도 점수가 아니라 문서 본문을 확인해야 판단할 수
  있다.


## 4. 실행 구조

```text
sample_report5.docx → 섹션 문서 → Document 변환
   │
   └─ vector_store.add_documents()
            │
            └─ search(query, k, metadata)
                  └─ vector_store.similarity_search_with_relevance_scores()
                        ├─ Metadata Filter
                        ├─ 질문 Embedding과 유사도 검색
                        └─ 정렬된 Top-k 반환
               ↓
      build_result_dataframe() → 검색 결과 표
               ↓
      outputs/retrieval/work_sample_report5_retrieval_results.csv 저장
```


## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [1]:
import re
import zipfile
from xml.etree import ElementTree as ET
from langchain_core.documents import Document

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_embedding_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import CHROMA_DIR, OUTPUT_DIR, PROJECT_ROOT, data_path
from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store
from agentic_ai.tools import search_keyword

settings = get_settings()
print_environment_summary(settings, needs_embedding_model=True)


[환경 설정 확인]
- 프로젝트: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks
- 데이터: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\data
- 출력: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\outputs
- OPENAI_API_KEY: 설정됨
- Embedding Model: text-embedding-3-small


## 6. 유사도 점수 먼저 읽기

이 실습에서는 코사인 유사도 공식을 직접 구현하지 않는다. 검색 결과에서 점수가 클수록
질문과 문서가 상대적으로 더 유사해 높은 순위를 받는다는 점에 집중한다. 아래 예시처럼
같은 후보군 안에서 점수를 비교해 순서를 정하며, 점수 자체는 정답 확률을 의미하지 않는다.


In [2]:
score_examples = [
    {"후보": "문서 A", "유사도": 0.71},
    {"후보": "문서 B", "유사도": 0.92},
    {"후보": "문서 C", "유사도": 0.34},
]

ranked_examples = sorted(score_examples, key=lambda item: item["유사도"], reverse=True)
for rank, example in enumerate(ranked_examples, start=1):
    print(f"{rank}위 {example['후보']}: {example['유사도']:.2f}")

print("주의: 가장 높은 점수도 '정답일 확률'은 아닙니다.")


1위 문서 B: 0.92
2위 문서 A: 0.71
3위 문서 C: 0.34
주의: 가장 높은 점수도 '정답일 확률'은 아닙니다.


## 7. 단계별 구현

### 7.1 DOCX 보고서 문서 생성

`sample_report5.docx`를 직접 로드해 문서 전면부와 Word Heading 1/2 기준의
`REPORT_DOCUMENTS`를 만든다. 여기에 Metadata Filter, 동의어, 동일 키워드의 다른
의미를 비교하기 위한 대조 문서 3개를 추가한다. 문서 수와 ID는 하드코딩하지 않고
실제 추출 결과에서 계산한다.


In [3]:
DATA_PATH = data_path("samples", "sample_report5.docx", must_exist=True)
DOCUMENT_ID = "sample-report-05"
COLLECTION_NAME = "work_sample_report5_v1"

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
NS = {"w": W_NS}


def load_docx_blocks(file_path) -> str:
    """DOCX의 문단·제목·표를 문서에 나타난 순서대로 텍스트로 변환한다."""
    with zipfile.ZipFile(file_path) as archive:
        root = ET.fromstring(archive.read("word/document.xml"))

    body = root.find("w:body", NS)
    if body is None:
        raise ValueError(f"DOCX 본문을 찾을 수 없습니다: {file_path}")

    blocks: list[str] = []
    for child in body:
        tag = child.tag.rsplit("}", 1)[-1]
        if tag == "p":
            text = "".join(node.text or "" for node in child.findall(".//w:t", NS)).strip()
            if not text:
                continue
            style_node = child.find("./w:pPr/w:pStyle", NS)
            style = style_node.get(f"{{{W_NS}}}val", "") if style_node is not None else ""
            if style == "Heading1":
                text = f"# {text}"
            elif style == "Heading2":
                text = f"## {text}"
            blocks.append(text)
        elif tag == "tbl":
            table_rows: list[str] = []
            for row in child.findall("./w:tr", NS):
                cells = []
                for cell in row.findall("./w:tc", NS):
                    cell_text = " ".join(
                        "".join(node.text or "" for node in paragraph.findall(".//w:t", NS)).strip()
                        for paragraph in cell.findall("./w:p", NS)
                    ).strip()
                    cells.append(cell_text)
                table_rows.append(" | ".join(cells))
            if table_rows:
                blocks.append("\n".join(table_rows))
    return "\n\n".join(blocks)


_DOCX_HEADING_RE = re.compile(r"^(#{1,2})\s+(.+)$", re.MULTILINE)


def build_report_documents(document_text: str) -> list[dict]:
    """DOCX의 전면부와 Heading 1/2 섹션을 검색 문서로 변환한다."""
    text = document_text.strip()
    headings = list(_DOCX_HEADING_RE.finditer(text))
    sections: list[tuple[str, str]] = []

    if not headings:
        sections.append(("전체 문서", text))
    else:
        front_matter = text[:headings[0].start()].strip()
        if front_matter:
            sections.append(("문서 메타데이터 및 종합 판단", front_matter))
        for index, match in enumerate(headings):
            end = headings[index + 1].start() if index + 1 < len(headings) else len(text)
            sections.append((match.group(2).strip(), text[match.start():end].strip()))

    return [
        {
            "doc_id": f"report5-{index:03d}",
            "document_id": DOCUMENT_ID,
            "section": section,
            "text": section_text,
        }
        for index, (section, section_text) in enumerate(sections, start=1)
    ]


document_text = load_docx_blocks(DATA_PATH)
REPORT_DOCUMENTS = build_report_documents(document_text)

# Metadata Filter와 의미 비교 실험용 대조 문서다. 보고서 문서와 ID 공간을 분리한다.
COMPARISON_DOCUMENTS = [
    {"doc_id": "compare-guide-01", "document_id": "guide-01", "section": "재택근무 가이드",
     "text": "재택근무 직원은 매일 오전 정기 화상 회의에 접속해 업무 현황을 공유해야 한다."},
    {"doc_id": "compare-notice-01", "document_id": "notice-01", "section": "사내 동호회",
     "text": "이번 분기 사내 등산 동호회 정기 모임은 다음 달 첫째 주 토요일에 진행된다."},
    {"doc_id": "compare-opinion-01", "document_id": "opinion-01", "section": "임원 의견",
     "text": "일부 임원진은 신규 리모트워크 정책 확대에 회의적인 시각을 보였다."},
]
SAMPLE_DOCUMENTS = REPORT_DOCUMENTS + COMPARISON_DOCUMENTS

print(f"입력 파일: {DATA_PATH}")
print(f"DOCX 추출 길이: {len(document_text):,}자")
print(f"보고서 섹션: {len(REPORT_DOCUMENTS)}개 / 대조 문서: {len(COMPARISON_DOCUMENTS)}개")
print(f"전체 검색 문서 수: {len(SAMPLE_DOCUMENTS)}")


입력 파일: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\data\samples\sample_report5.docx
DOCX 추출 길이: 3,846자
보고서 섹션: 12개 / 대조 문서: 3개
전체 검색 문서 수: 15


### 7.2 Vector Store 구성

문서를 `Document`로 변환해 Vector Store에 추가한다. 문서 Embedding 생성과 저장은
Vector Store가 담당하며, 애플리케이션에서 Embedding 목록을 직접 관리하지 않는다.

> 이 실습은 로컬 디스크의 `outputs/vectorstore/chroma/`에 Chroma DB를 저장한다.
> 노트북을 다시 실행해도 같은 컬렉션과 문서를 재사용하며, 동일한 문서 ID는 새로
> 중복 추가하지 않고 갱신한다.


In [4]:
embedding_model = get_embedding_model()
# 검색 대상 본문은 page_content에, 필터·출처 정보는 metadata에 분리해 저장한다.
VECTOR_DOCUMENTS = [
    Document(
        page_content=record["text"],
        metadata={key: value for key, value in record.items() if key != "text"},
    )
    for record in SAMPLE_DOCUMENTS
]

vector_store = get_chroma_store(
    COLLECTION_NAME,
    embedding_model=embedding_model,
)
# 영속 Collection이 비어 있을 때만 추가해 Notebook 재실행 시 중복 적재를 막는다.
documents_added, document_count = add_documents_if_empty(
    vector_store,
    VECTOR_DOCUMENTS,
    ids=[record["doc_id"] for record in SAMPLE_DOCUMENTS],
)

action = "초기화" if documents_added else "기존 Collection 재사용"
print(f"Chroma Collection: {COLLECTION_NAME}")
print(f"디스크 저장 경로: {CHROMA_DIR}")
print(f"처리 결과: {action} ({document_count}개 문서)")


Chroma Collection: work_sample_report5_v1
디스크 저장 경로: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\outputs\vectorstore\chroma
처리 결과: 초기화 (15개 문서)


### 7.3 유사도 검색

질문을 문자열로 전달하면 Vector Store가 질문 Embedding, 유사도 계산, 정렬, Top-k
선택을 처리한다. 반환값은 `(Document, score)` 튜플 목록이다.


In [5]:
# relevance score는 후보 간 순위를 위한 값이며 정답일 확률로 해석하면 안 된다.
sample_matches = vector_store.similarity_search_with_relevance_scores(
    "2026년 상반기 리모트워크 만족도는 어땠나요?",
    k=3,
)
for document, score in sample_matches:
    print(f"score={score:.4f} | {document.metadata['section']} | {document.page_content}")


score=0.5614 | 문서 메타데이터 및 종합 판단 | 2026 MID-YEAR PEOPLE OPERATIONS REPORT

2026년 사내 리모트워크운영 현황 보고서

상반기 운영 성과, 리스크 및 하반기 개선 계획

기준 기간 | 2026. 1. 1. - 6. 30. | 작성일 | 2026. 7. 15.
작성 부서 | 인사운영팀 | 문서 등급 | 내부용

종합 판단  리모트워크는 안정화 단계에 진입했다. 다만 부서 간 활용 격차와 신입 온보딩 품질을 하반기 핵심 통제 과제로 관리할 필요가 있다.
score=0.4037 | 경영진 요약 | # 경영진 요약

핵심 지표 | 2026년 상반기, 전년 대비

정기 이용률 | 제도 만족도 | 관리자 만족도 | 협업 지연 경험
68%  ▲ 6%p | 78%  ▲ 4%p | 72%  ▲ 5%p | 32%  ▼ 9%p
score=0.3864 | 2. 만족도 및 협업 효과 | # 2. 만족도 및 협업 효과

6월 전사 설문에는 212명이 참여해 88%의 응답률을 기록했다. 전반 만족도는 상승했으나, 소통 경험과 장비 지원은 개선 여지가 남아 있다.

표 3. 경험 항목별 긍정 응답률

평가 항목 | 긍정 응답 | 전년 대비 | 판단
출퇴근 부담 감소 | 88% | +2%p | 강점 유지
업무 자율성 | 84% | +3%p | 강점 유지
집중 업무 환경 | 82% | +5%p | 개선 확인
제도 전반 만족 | 78% | +4%p | 목표 상회
장비·IT 지원 | 71% | +6%p | 추가 보완
팀 커뮤니케이션 | 69% | +4%p | 우선 개선
화상회의 피로 관리 | 64% | +7%p | 우선 개선

협업 불편 변화  의사결정·피드백 지연 35%(-6%p), 화상회의 피로 26%(-1%p), 자료·결정 내용 탐색 21%(-1%p), 장비·접속 환경 18%(-1%p)로 모두 개선됐다.


### 7.4 검색 결과 변환

Vector Store가 반환한 `Document`와 점수를 실습에서 관찰하기 쉬운 dict 형태로
변환하고, 반환 순서대로 순위를 부여한다.


In [6]:
def to_result_records(matches: list[tuple[Document, float]]) -> list[dict]:
    """Vector Store 검색 결과를 순위가 포함된 dict 목록으로 변환한다."""
    # Vector Store가 유사도순으로 반환한 순서를 유지하며 1부터 rank를 붙인다.
    return [
        {
            **document.metadata,
            "text": document.page_content,
            "similarity": score,
            "rank": rank,
        }
        for rank, (document, score) in enumerate(matches, start=1)
    ]


### 7.5 Metadata Filter

Chroma의 `filter` 인자에 전달할 metadata 조건 dict를 만든다. 필터는 유사도 검색 전에
DB 내부 후보 문서를 좁히며, 점수 계산과 정렬은 Chroma가 담당한다.


In [7]:
def make_metadata_filter(
    document_id: str | None = None,
    section: str | None = None,
) -> dict | None:
    """Chroma에 전달할 metadata 필터를 만든다."""
    # 선택된 조건만 모아 필터가 없는 경우와 단일·복합 조건을 구분한다.
    conditions = []
    if document_id is not None:
        conditions.append({"document_id": document_id})
    if section is not None:
        conditions.append({"section": section})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    # 여러 metadata 조건은 모두 만족해야 하므로 Chroma의 $and 연산자로 묶는다.
    return {"$and": conditions}


### 7.6 Vector Store 검색 함수

`search()`는 조건과 `k`를 Vector Store에 전달하고 결과 형식만 변환한다.
Embedding 생성, 유사도 계산, 정렬, Top-k 선택은 직접 구현하지 않는다.


In [8]:
def search(
    query: str,
    k: int = 3,
    document_id: str | None = None,
    section: str | None = None,
) -> list[dict]:
    """Vector Store의 유사도 검색과 Metadata Filter를 실행한다."""
    if k < 1:
        raise ValueError("k는 1 이상이어야 합니다.")
    # 필터는 검색 후 결과를 지우는 것이 아니라 유사도를 계산할 후보군 자체를 제한한다.
    metadata_filter = make_metadata_filter(document_id=document_id, section=section)
    matches = vector_store.similarity_search_with_relevance_scores(
        query,
        k=k,
        filter=metadata_filter,
    )
    return to_result_records(matches)


### 7.7 Metadata Filter 실행

`document_id` 조건을 Vector Store 검색에 전달해 후보 문서가 실제로 좁혀지는지
확인한다.


In [9]:
filtered_demo = search(
    "리모트워크",
    k=len(SAMPLE_DOCUMENTS),
    document_id=DOCUMENT_ID,
)
print(f"document_id={DOCUMENT_ID!r} 필터 적용 후 결과 수: {len(filtered_demo)} / 전체 {len(SAMPLE_DOCUMENTS)}")
print({result["document_id"] for result in filtered_demo})


document_id='sample-report-05' 필터 적용 후 결과 수: 12 / 전체 15
{'sample-report-05'}


### 7.8 검색 결과 DataFrame 출력

Vector Store 검색 결과를 DataFrame으로 변환해 순위, metadata, 유사도 점수와 본문을
한눈에 비교한다.


In [10]:
import pandas as pd
def build_result_dataframe(query: str, results: list[dict]) -> pd.DataFrame:
    """검색 결과를 표 형식으로 만든다."""
    rows = [
        {
            "query": query,
            "rank": r["rank"],
            "doc_id": r["doc_id"],
            "document_id": r["document_id"],
            "section": r["section"],
            "similarity": round(r["similarity"], 4),
            "text": r["text"],
        }
        for r in results
    ]
    return pd.DataFrame(rows)


baseline_results = search("2026년 상반기 리모트워크 만족도는 어땠나요?", k=3)
build_result_dataframe("2026년 상반기 리모트워크 만족도는 어땠나요?", baseline_results)


,query,rank,doc_id,document_id,section,similarity,text
0,2026년 상반기 리모트워크 만족도는 어땠나요?,1,report5-001,sample-report-05,문서 메타데이터 및 종합 판단,0.5614,2026 MID-YEAR PEOPLE OPERATIONS REPORT\n\n2026...
1,2026년 상반기 리모트워크 만족도는 어땠나요?,2,report5-002,sample-report-05,경영진 요약,0.4037,"# 경영진 요약\n\n핵심 지표 | 2026년 상반기, 전년 대비\n\n정기 이용률..."
2,2026년 상반기 리모트워크 만족도는 어땠나요?,3,report5-006,sample-report-05,2. 만족도 및 협업 효과,0.3864,# 2. 만족도 및 협업 효과\n\n6월 전사 설문에는 212명이 참여해 88%의 ...


## 8. 실행 결과 관찰

`baseline_results`의 순위와 유사도 점수, 그리고 실제 본문을 함께 확인한다.


In [11]:
for r in baseline_results:
    print(f"[{r['rank']}위] similarity={r['similarity']:.4f} | section={r['section']} | {r['text']}")


[1위] similarity=0.5614 | section=문서 메타데이터 및 종합 판단 | 2026 MID-YEAR PEOPLE OPERATIONS REPORT

2026년 사내 리모트워크운영 현황 보고서

상반기 운영 성과, 리스크 및 하반기 개선 계획

기준 기간 | 2026. 1. 1. - 6. 30. | 작성일 | 2026. 7. 15.
작성 부서 | 인사운영팀 | 문서 등급 | 내부용

종합 판단  리모트워크는 안정화 단계에 진입했다. 다만 부서 간 활용 격차와 신입 온보딩 품질을 하반기 핵심 통제 과제로 관리할 필요가 있다.
[2위] similarity=0.4037 | section=경영진 요약 | # 경영진 요약

핵심 지표 | 2026년 상반기, 전년 대비

정기 이용률 | 제도 만족도 | 관리자 만족도 | 협업 지연 경험
68%  ▲ 6%p | 78%  ▲ 4%p | 72%  ▲ 5%p | 32%  ▼ 9%p
[3위] similarity=0.3864 | section=2. 만족도 및 협업 효과 | # 2. 만족도 및 협업 효과

6월 전사 설문에는 212명이 참여해 88%의 응답률을 기록했다. 전반 만족도는 상승했으나, 소통 경험과 장비 지원은 개선 여지가 남아 있다.

표 3. 경험 항목별 긍정 응답률

평가 항목 | 긍정 응답 | 전년 대비 | 판단
출퇴근 부담 감소 | 88% | +2%p | 강점 유지
업무 자율성 | 84% | +3%p | 강점 유지
집중 업무 환경 | 82% | +5%p | 개선 확인
제도 전반 만족 | 78% | +4%p | 목표 상회
장비·IT 지원 | 71% | +6%p | 추가 보완
팀 커뮤니케이션 | 69% | +4%p | 우선 개선
화상회의 피로 관리 | 64% | +7%p | 우선 개선

협업 불편 변화  의사결정·피드백 지연 35%(-6%p), 화상회의 피로 26%(-1%p), 자료·결정 내용 탐색 21%(-1%p), 장비·접속 환경 18%(-1%p)로 모두 개선됐다.


**결과 해석**: 순위(`rank`)는 유사도 점수(`similarity`)를 기준으로 매겨진다. 점수
자체의 절대값보다는, 어떤 문서가 다른 문서보다 상대적으로 질문과 가까운지를
비교하는 데 의미가 있다.


## 9. 비교 실험

### 9.1 Top-1과 Top-3


In [12]:
query_a = "화상회의 피로도를 줄이는 방법은?"
top1 = search(query_a, k=1)
top3 = search(query_a, k=3)
print("Top-1:")
for r in top1:
    print(f"  [{r['rank']}위] {r['section']} | {r['text']}")
print("Top-3:")
for r in top3:
    print(f"  [{r['rank']}위] {r['section']} | {r['text']}")


Top-1:
  [1위] 집중 진단 2 | 하이브리드 회의 격차 | ## 집중 진단 2 | 하이브리드 회의 격차

참여자의 29%가 발언 순서와 화면 밖 논의를 따라가기 어렵다고 답했다. 결정 사항이 남지 않으면 후속 질의가 평균 1.8배 늘었다.

보완 방향  진행자와 기록자를 지정하고, 종료 2시간 이내 결정·담당자·기한을 공용 공간에 게시한다.

통제 원칙  직무 특성과 팀 협업 리듬을 기준으로 원칙을 설명하고, 개인 활동량이 아닌 보안·성과·경험 지표를 월 단위로 관리한다. 문제 발생 시 원격 일수보다 프로세스·역할·도구를 먼저 개선한다.

04  H2 ACTION PLAN
Top-3:
  [1위] 집중 진단 2 | 하이브리드 회의 격차 | ## 집중 진단 2 | 하이브리드 회의 격차

참여자의 29%가 발언 순서와 화면 밖 논의를 따라가기 어렵다고 답했다. 결정 사항이 남지 않으면 후속 질의가 평균 1.8배 늘었다.

보완 방향  진행자와 기록자를 지정하고, 종료 2시간 이내 결정·담당자·기한을 공용 공간에 게시한다.

통제 원칙  직무 특성과 팀 협업 리듬을 기준으로 원칙을 설명하고, 개인 활동량이 아닌 보안·성과·경험 지표를 월 단위로 관리한다. 문제 발생 시 원격 일수보다 프로세스·역할·도구를 먼저 개선한다.

04  H2 ACTION PLAN
  [2위] 2. 만족도 및 협업 효과 | # 2. 만족도 및 협업 효과

6월 전사 설문에는 212명이 참여해 88%의 응답률을 기록했다. 전반 만족도는 상승했으나, 소통 경험과 장비 지원은 개선 여지가 남아 있다.

표 3. 경험 항목별 긍정 응답률

평가 항목 | 긍정 응답 | 전년 대비 | 판단
출퇴근 부담 감소 | 88% | +2%p | 강점 유지
업무 자율성 | 84% | +3%p | 강점 유지
집중 업무 환경 | 82% | +5%p | 개선 확인
제도 전반 만족 | 78% | +4%p | 목표 상회
장비·IT 지원 | 71% | +6%p | 추가 보완
팀 커뮤니케이션 | 69% | +4%p |

**관찰**: Top-1만 보면 가장 유사도가 높은 문서 하나만 확인할 수 있지만, 실제 정답에
필요한 정보가 2~3위 문서에 나뉘어 있을 수도 있다. Top-k를 늘리면 놓칠 수 있는 정보를
줄일 수 있지만, 그만큼 무관한 문서가 섞일 가능성도 커진다.


### 9.2 짧은 검색어와 구체적인 검색어


In [13]:
# 같은 주제라도 구체적인 검색어가 더 많은 의미 단서를 제공해 순위를 바꿀 수 있다.
short_query = "회의"
specific_query = "화상회의 피로도를 낮추기 위한 회의 시간 단축 방안"

short_results = search(short_query, k=3)
specific_results = search(specific_query, k=3)

print(f"짧은 검색어 '{short_query}':")
for r in short_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | {r['section']}")
print(f"구체적인 검색어 '{specific_query}':")
for r in specific_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | {r['section']}")


짧은 검색어 '회의':
  [1위] similarity=0.3320 | 집중 진단 2 | 하이브리드 회의 격차
  [2위] similarity=0.2906 | 사내 동호회
  [3위] similarity=0.2873 | 재택근무 가이드
구체적인 검색어 '화상회의 피로도를 낮추기 위한 회의 시간 단축 방안':
  [1위] similarity=0.5091 | 집중 진단 2 | 하이브리드 회의 격차
  [2위] similarity=0.4042 | 재택근무 가이드
  [3위] similarity=0.3789 | 하반기 우선 조치


**관찰**: 짧은 검색어는 여러 문서와 폭넓게 비슷하게 나올 수 있어 순위 사이의 점수
차이가 작을 수 있다. 구체적인 검색어는 관련 문서와 무관한 문서 사이의 유사도 차이가
더 뚜렷하게 나타나는 경향이 있다.


### 9.3 동의어


In [14]:
original_term_query = "리모트워크 직원의 화상 회의 참여 방식"
synonym_query = "재택근무 직원의 화상 회의 참여 방식"

original_results = search(original_term_query, k=3)
synonym_results = search(synonym_query, k=3)

print("'리모트워크' 검색 결과:")
for r in original_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | doc_id={r['doc_id']} | {r['section']}")
print("'재택근무'(동의어) 검색 결과:")
for r in synonym_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | doc_id={r['doc_id']} | {r['section']}")


'리모트워크' 검색 결과:
  [1위] similarity=0.4745 | doc_id=report5-010 | 집중 진단 2 | 하이브리드 회의 격차
  [2위] similarity=0.4535 | doc_id=compare-guide-01 | 재택근무 가이드
  [3위] similarity=0.4360 | doc_id=report5-012 | 운영 가이드라인
'재택근무'(동의어) 검색 결과:
  [1위] similarity=0.7205 | doc_id=compare-guide-01 | 재택근무 가이드
  [2위] similarity=0.5252 | doc_id=report5-010 | 집중 진단 2 | 하이브리드 회의 격차
  [3위] similarity=0.4714 | doc_id=report5-006 | 2. 만족도 및 협업 효과


**관찰**: 키워드 검색은 정확한 표면 문자열만 찾지만 Embedding 검색은
`"리모트워크"`와 `"재택근무"`처럼 표현이 달라도 문맥이 가까운 문서를 후보로 올릴 수
있다. 실제 순위와 점수는 사용하는 Embedding 모델과 전체 문서 집합에 따라 달라지므로
고정된 문서 ID나 점수를 정답처럼 가정하지 않고 결과 본문을 함께 확인한다.


### 9.4 동일 키워드의 다른 의미


In [15]:
# 같은 표면 문자열 "회의"가 모임과 의심이라는 다른 의미로 쓰인 문서를 비교한다.
meeting_doc = next(record for record in REPORT_DOCUMENTS if "화상회의 피로" in record["text"])
skeptical_doc = next(record for record in COMPARISON_DOCUMENTS if "회의적인" in record["text"])

meeting_keyword_hit = search_keyword(meeting_doc["text"], "회의")
skeptical_keyword_hit = search_keyword(skeptical_doc["text"], "회의")
print("키워드 검색('회의') 결과:")
print(f"  화상회의 피로 문서: found={meeting_keyword_hit['found']}")
print(f"  회의적인 시각 문서: found={skeptical_keyword_hit['found']}")

meeting_query = "회의 때문에 피곤한 이유"
meeting_results = search(meeting_query, k=len(SAMPLE_DOCUMENTS))
meeting_rank = next(result["rank"] for result in meeting_results if result["doc_id"] == meeting_doc["doc_id"])
skeptical_rank = next(result["rank"] for result in meeting_results if result["doc_id"] == skeptical_doc["doc_id"])
print(f"{meeting_query!r} 질문에서 화상회의 문서 순위: {meeting_rank}, 회의적 의견 문서 순위: {skeptical_rank}")


키워드 검색('회의') 결과:
  화상회의 피로 문서: found=True
  회의적인 시각 문서: found=True
'회의 때문에 피곤한 이유' 질문에서 화상회의 문서 순위: 3, 회의적 의견 문서 순위: 11


**관찰**: 키워드 검색은 `"회의"`라는 글자만 보고 화상회의 피로 문서와
회의적인 의견 문서를 모두 "찾음"으로 판단한다. 그러나 Embedding 검색은 질문의
문맥을 사용하므로 `"회의 때문에 피곤한 이유"`에는 화상회의 관련 보고서 섹션이 더
적절한 후보인지 순위와 본문을 통해 비교할 수 있다.


### 9.5 Metadata Filter 적용 전후


In [16]:
query_b = "정기 모임 일정이 궁금해요"
without_filter = search(query_b, k=3)
with_filter = search(query_b, k=3, document_id=DOCUMENT_ID)

print(f"Filter 미적용 (전체 {len(SAMPLE_DOCUMENTS)}개 문서가 후보):")
for result in without_filter:
    print(f"  [{result['rank']}위] document_id={result['document_id']} | {result['section']}")
print(f"Filter 적용 (document_id={DOCUMENT_ID!r}만 후보, {len(REPORT_DOCUMENTS)}개):")
for result in with_filter:
    print(f"  [{result['rank']}위] document_id={result['document_id']} | {result['section']}")


Filter 미적용 (전체 15개 문서가 후보):
  [1위] document_id=sample-report-05 | 집중 진단 2 | 하이브리드 회의 격차
  [2위] document_id=guide-01 | 재택근무 가이드
  [3위] document_id=opinion-01 | 임원 의견
Filter 적용 (document_id='sample-report-05'만 후보, 12개):
  [1위] document_id=sample-report-05 | 집중 진단 2 | 하이브리드 회의 격차
  [2위] document_id=sample-report-05 | 운영 가이드라인
  [3위] document_id=sample-report-05 | 하반기 우선 조치


**관찰**: `"정기 모임"`은 별도 사내 동호회 대조 문서와 유사도가 높을 수
있다. Filter 없이 검색하면 보고서와 무관한 결과가 섞일 수 있지만,
`document_id=DOCUMENT_ID`로 제한하면 `sample_report5.docx`에서 추출한 섹션만 후보가
된다. Metadata Filter는 유사도를 계산하기 전에 후보군 자체를 바꾼다는 점이 핵심이다.


## 10. 실패 실험과 교정: 유사도 점수를 정답 확률로 오해하기

실제 문서 대신 통제된 검색 결과를 사용해, Vector Store가 반환한 유사도 점수가 높아도
그 문서가 질문의 정답을 담고 있다고 보장할 수 없음을 확인한다.


In [17]:
controlled_results = [
    {"doc_id": "A", "similarity": 0.95, "contains_answer": False},
    {"doc_id": "B", "similarity": 0.78, "contains_answer": True},
]

top_result = max(controlled_results, key=lambda item: item["similarity"])
print("가장 높은 점수의 문서:", top_result)
print("실제 정답 포함 여부:", top_result["contains_answer"])
print("→ 유사도 점수는 후보 순위를 정할 뿐, 정답 여부는 본문이나 근거 평가로 확인해야 한다.")


가장 높은 점수의 문서: {'doc_id': 'A', 'similarity': 0.95, 'contains_answer': False}
실제 정답 포함 여부: False
→ 유사도 점수는 후보 순위를 정할 뿐, 정답 여부는 본문이나 근거 평가로 확인해야 한다.


**교정된 접근**: 유사도 점수를 "정답 확률"로 취급해 자동으로 채택하지 않는다. 대신
Top-k로 후보를 좁힌 뒤, 각 후보의 실제 본문을 확인하거나(사람 또는 LLM), Notebook 03-3에서
다룰 "근거 적합성 평가" 단계를 거쳐 정말로 질문에 답할 수 있는 내용인지 검증해야 한다.
8~9번에서 얻은 검색 결과도 순위와 점수만 보지 말고, `text` 필드를 직접 읽어 질문에
대한 답이 맞는지 확인하는 습관이 필요하다.


## 11. 도전 과제

1. `SAMPLE_DOCUMENTS`에 완전히 새로운 주제의 문서를 2~3개 추가하고, 관련 없는
   질문을 던졌을 때 유사도 점수가 어떻게 분포하는지 관찰한다.
2. `search()`에 `section` 필터까지 함께 적용해, `document_id`와 `section`을 동시에
   좁혔을 때 후보 수가 어떻게 줄어드는지 확인한다.
3. Top-k를 1부터 9까지 바꿔가며 같은 질문을 검색하고, 몇 번째 순위부터 유사도
   점수가 급격히 낮아지는지(문서와 무관해지는지) 관찰한다.


## 12. 테스트

**테스트 유형: 외부 API 통합 테스트 — OpenAI Embedding + Chroma**

검색 순위와 metadata filter를 검증한다. 실패하면 코드와 함께 API Key, 네트워크, Embedding 모델과 Chroma Collection 상태를 확인한다.


In [18]:
assert len(REPORT_DOCUMENTS) >= 5
search_check = search("2026년 상반기 리모트워크 만족도", k=3, document_id=DOCUMENT_ID)
assert len(search_check) == 3
assert [result["rank"] for result in search_check] == [1, 2, 3]
assert all(
    search_check[index]["similarity"] >= search_check[index + 1]["similarity"]
    for index in range(len(search_check) - 1)
)
assert all(result["document_id"] == DOCUMENT_ID for result in search_check)

all_filtered_check = search("리모트워크", k=len(SAMPLE_DOCUMENTS), document_id=DOCUMENT_ID)
assert len(all_filtered_check) == len(REPORT_DOCUMENTS)
assert all(result["document_id"] == DOCUMENT_ID for result in all_filtered_check)

try:
    search("리모트워크", k=0)
    raise AssertionError("k=0이 통과했습니다.")
except ValueError:
    pass

print("DOCX 적재·Embedding 검색 테스트 통과")


DOCX 적재·Embedding 검색 테스트 통과


## 13. 결과 저장


In [19]:
all_result_rows = []
for label, results in [
    ("baseline", baseline_results),
    ("top1_top3_top1", top1),
    ("top1_top3_top3", top3),
    ("short_query", short_results),
    ("specific_query", specific_results),
    ("original_term", original_results),
    ("synonym", synonym_results),
    ("without_filter", without_filter),
    ("with_filter", with_filter),
]:
    df = build_result_dataframe(label, results)
    all_result_rows.extend(df.to_dict(orient="records"))

retrieval_results_df = pd.DataFrame(all_result_rows)
retrieval_dir = OUTPUT_DIR / "retrieval"
retrieval_dir.mkdir(parents=True, exist_ok=True)
retrieval_csv_path = retrieval_dir / "work_sample_report5_retrieval_results.csv"
retrieval_results_df.to_csv(retrieval_csv_path, index=False, encoding="utf-8-sig")
print("저장 위치:", retrieval_csv_path)

embedding_log = {
    "document_count": len(SAMPLE_DOCUMENTS),
    "vector_store": type(vector_store).__name__,
    "baseline_query": "2026년 상반기 리모트워크 만족도는 어땠나요?",
    "baseline_top_results": [
        {"rank": r["rank"], "doc_id": r["doc_id"], "similarity": round(r["similarity"], 4)}
        for r in baseline_results
    ],
}
saved_path = save_log(embedding_log, OUTPUT_DIR / "logs" / "work_2_embedding_retrieval_log.json")
print("저장 위치:", saved_path)


저장 위치: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\outputs\retrieval\work_sample_report5_retrieval_results.csv
저장 위치: E:\future_lab\agentic_ai_hrd\agentic-ai-notebooks\outputs\logs\work_2_embedding_retrieval_log.json


## 14. 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store는 문서 Embedding 저장, 질문 Embedding, 유사도 계산, 정렬, Top-k
  선택을 담당하며 애플리케이션은 검색 조건과 결과 활용에 집중한다.
- 유사도 점수는 순위를 매기는 상대적 신호일 뿐, 정답일 확률이 아니다.
- `k`는 Vector Store가 반환할 상위 결과 수를 정하고, Metadata Filter는 유사도 검색
  전에 후보군 자체를 좁힌다.
- 짧은 검색어, 동의어, 동일 키워드의 다른 의미는 모두 검색 결과에 영향을 주며,
  키워드 검색과 Embedding 검색이 서로 다르게 반응한다.
- 검색 결과를 그대로 신뢰하지 않고, 실제 본문을 확인해 정말 질문에 답이 되는지
  검증하는 절차가 필요하다.


## 15. 확인 문제

1. 유사도 점수 0.85가 "정답일 확률 85%"를 의미하지 않는 이유는 무엇인가?
2. 순위 정렬/Top-k와 Metadata Filter는 검색 후보군에 어떻게 다르게 영향을 주는가?
3. 키워드 검색이 "재택근무"와 "리모트워크"를 같은 의미로 찾지 못하는 이유는
   무엇인가?
4. 유사도 점수가 높은 검색 결과를 받았을 때, 그것을 바로 정답으로 채택하면 안 되는
   이유는 무엇이며 대신 어떻게 해야 하는가?
